In [6]:
from pathlib import Path
import numpy as np
import pypulseq as pp
from bmc.utils.seq.write import write_seq

In [7]:
wdir = Path().resolve().parent
folder = wdir / "seq_lib"

sys = pp.Opts(
    max_grad=500,
    grad_unit="mT/m",
    max_slew=1e9,
    slew_unit="T/m/s",
    rf_ringdown_time=0,
    rf_dead_time=0,
    rf_raster_time=1e-6,
    gamma=42576400,
    grad_raster_time=1e-6,
)

AUTHOR = "DANIEL MIKSCH"
FLAG_CHECK_TIMING = True

### Definitions

In [8]:
seqid = "zspec_block"  # file name

defs: dict = {}
defs["b0"]        = 17        # B0 field strength [T]
defs["m0_offset"] = 300       # M0 reference offset [ppm] (far off-resonance)
defs["n_ETM"]     = 1

# saturation pulse  (1 µT at 17T ≈ 42.6 Hz  →  3 µT ≈ 128 Hz)
b1_sat_hz   = 128.0           # B1 amplitude [Hz]  (~3 µT at 17T)
t_sat       = 2.0             # saturation duration [s]  (>> T1 ≈ 2.5 s)

# offsets
offset_range = 10             # [ppm]
n_offsets    = 200            # number of offset points

pseudo_adc = pp.make_adc(num_samples=1, duration=1e-3)

GAMMA_HZ     = sys.gamma * 1e-6          # [MHz/T], sys.gamma in Hz/T
defs["freq"] = defs["b0"] * GAMMA_HZ    # Larmor frequency [MHz] — used as Hz/ppm in offset calc

offsets_ppm = np.append(
    defs["m0_offset"],
    np.linspace(-offset_range, offset_range, n_offsets),
)
defs["offsets_ppm"] = offsets_ppm
defs["num_meas"]    = offsets_ppm.size
defs["seq_id_string"] = seqid
seq_filename = seqid + ".seq"

print(f"Larmor frequency : {defs['freq'] * 1e6 / 1e6:.2f} MHz")
print(f"B1 amplitude     : {b1_sat_hz:.1f} Hz  ({b1_sat_hz / 42.576:.2f} µT at 17T)")
print(f"Saturation time  : {t_sat:.1f} s")
print(f"Offsets          : {offsets_ppm.size} points  ({offsets_ppm[1]:.1f} to {offsets_ppm[-1]:.1f} ppm)")

Larmor frequency : 723.80 MHz
B1 amplitude     : 128.0 Hz  (3.01 µT at 17T)
Saturation time  : 2.0 s
Offsets          : 201 points  (-10.0 to 10.0 ppm)


### Build sequence

In [9]:
seq = pp.Sequence()
offsets_hz = offsets_ppm * defs["freq"]

for offset_hz in offsets_hz:
    # block saturation pulse
    rf_sat = pp.make_block_pulse(
        flip_angle=b1_sat_hz * t_sat * 2 * np.pi,
        system=sys,
        duration=t_sat,
        freq_offset=offset_hz,
        phase_offset=0,
    )
    seq.add_block(rf_sat)
    seq.add_block(pseudo_adc)

if FLAG_CHECK_TIMING:
    ok, error_report = seq.check_timing()
    if ok:
        print("Timing check passed")
    else:
        print("Timing check failed:")
        print(error_report)

Timing check passed


In [10]:
write_seq(seq=seq, seq_defs=defs, filename=folder / seq_filename, author=AUTHOR, use_matlab_names=True)
print(f"Saved: {folder / seq_filename}")

Saved: /Users/danielmiksch/JupyterLab/optim/seq_lib/zspec_block.seq
